# **Food Price Merge with Master Modeling Dataset**

This notebook merges the processed food price features with the existing IPC + rainfall + NDVI master modeling dataset.

The previous notebook created two food price feature files:

- `county_food_price_monthly_features.csv`
- `national_food_price_monthly_features.csv`

The county-level file contains food price features for counties with mapped market coverage.

The national-level file contains Kenya-wide monthly food price features that can be used as proxy indicators for counties without direct county-level market coverage.

The goal of this notebook is to create a new modeling dataset that includes:

- IPC food insecurity outcomes
- CHIRPS rainfall features
- MODIS NDVI features
- County-level food price features where available
- National food price proxy features for all county-period records

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
RAW_DIR = Path("../02_data/raw")
PROCESSED_DIR = Path("../02_data/processed")

# Input files
MASTER_FILE = PROCESSED_DIR / "ipc_rainfall_ndvi_master_dataset.csv"
COUNTY_PRICE_FILE = PROCESSED_DIR / "county_food_price_monthly_features.csv"
NATIONAL_PRICE_FILE = PROCESSED_DIR / "national_food_price_monthly_features.csv"

# Output file
OUTPUT_FILE = PROCESSED_DIR / "ipc_rainfall_ndvi_food_price_master_dataset.csv"

## **Load Input Datasets**

This section loads the existing master modeling dataset and the processed food price feature files.

The master dataset is the main modeling table already created in earlier notebooks. It contains IPC food insecurity outcomes, rainfall features, and NDVI features.

The food price feature files were created in Notebook 12.

In [3]:
# Load datasets

master_df = pd.read_csv(MASTER_FILE)
county_prices = pd.read_csv(COUNTY_PRICE_FILE)
national_prices = pd.read_csv(NATIONAL_PRICE_FILE)

print("Master dataset shape:", master_df.shape)
print("County price features shape:", county_prices.shape)
print("National price features shape:", national_prices.shape)

display(master_df.head())
display(county_prices.head())
display(national_prices.head())

Master dataset shape: (322, 19)
County price features shape: (1448, 16)
National price features shape: (244, 13)


,ipc_date,analysis_period,county,max_ipc_phase,phase_3_plus_population,total_population,phase_3_plus_percentage,rainfall_date,mean_rainfall_mm,rainfall_3_month_total,rainfall_6_month_total,rainfall_3_month_avg,rainfall_6_month_avg,date,mean_ndvi,ndvi_1_month_mean,ndvi_3_month_mean,ndvi_6_month_mean,ndvi_anomaly
0,2019-07-01,Jul 2019,Baringo,4,105555,703697,0.15,2019-06-01,72.692665,143.441454,172.723779,47.813818,28.787296,2019-06-01,0.645498,0.645498,0.469621,0.424800,0.078739
1,2019-07-01,Jul 2019,Embu,3,32883,219220,0.15,2019-06-01,8.068131,123.946965,156.388124,41.315655,26.064687,2019-06-01,0.710059,0.710059,0.715939,0.708263,-0.026823
2,2019-07-01,Jul 2019,Garissa,4,151183,431950,0.35,2019-06-01,5.417682,55.185197,66.812339,18.395066,11.135390,2019-06-01,0.211246,0.211246,0.247841,0.247458,-0.047864
3,2019-07-01,Jul 2019,Isiolo,4,54413,155465,0.35,2019-06-01,0.936229,26.591658,40.062429,8.863886,6.677072,2019-06-01,0.207090,0.207090,0.223339,0.220767,-0.042075
4,2019-07-01,Jul 2019,Kajiado,3,43536,870721,0.05,2019-06-01,6.391582,62.716861,88.344720,20.905620,14.724120,2019-06-01,0.379489,0.379489,0.389526,0.366431,-0.030230


,county,month,county_beans_price_per_kg,county_maize_price_per_kg,county_rice_price_per_kg,county_staple_price_per_kg,county_price_available,county_staple_items_available,county_maize_price_per_kg_3_month_avg,county_maize_price_per_kg_6_month_avg,county_beans_price_per_kg_3_month_avg,county_beans_price_per_kg_6_month_avg,county_rice_price_per_kg_3_month_avg,county_rice_price_per_kg_6_month_avg,county_staple_price_per_kg_3_month_avg,county_staple_price_per_kg_6_month_avg
0,Baringo,2015-01-01,104.0,42.0,NaN,73.0,1,2,42.000000,42.000000,104.000000,104.000000,NaN,NaN,73.000000,73.000
1,Baringo,2015-02-01,107.0,47.0,NaN,77.0,1,2,44.500000,44.500000,105.500000,105.500000,NaN,NaN,75.000000,75.000
2,Baringo,2015-03-01,112.0,38.0,NaN,75.0,1,2,42.333333,42.333333,107.666667,107.666667,NaN,NaN,75.000000,75.000
3,Baringo,2015-04-01,105.0,42.0,NaN,73.5,1,2,42.333333,42.250000,108.000000,107.000000,NaN,NaN,75.166667,74.625
4,Baringo,2015-05-01,106.0,45.0,NaN,75.5,1,2,41.666667,42.800000,107.666667,106.800000,NaN,NaN,74.666667,74.800


,month,national_beans_price_per_kg,national_maize_price_per_kg,national_rice_price_per_kg,national_staple_price_per_kg,national_maize_price_per_kg_3_month_avg,national_maize_price_per_kg_6_month_avg,national_beans_price_per_kg_3_month_avg,national_beans_price_per_kg_6_month_avg,national_rice_price_per_kg_3_month_avg,national_rice_price_per_kg_6_month_avg,national_staple_price_per_kg_3_month_avg,national_staple_price_per_kg_6_month_avg
0,2006-01-01,38.587654,17.775833,NaN,28.181744,17.775833,17.775833,38.587654,38.587654,NaN,NaN,28.181744,28.181744
1,2006-02-01,41.259383,18.741019,NaN,30.000201,18.258426,18.258426,39.923519,39.923519,NaN,NaN,29.090972,29.090972
2,2006-03-01,45.745864,18.266917,NaN,32.006390,18.261256,18.261256,41.864300,41.864300,NaN,NaN,30.062778,30.062778
3,2006-04-01,46.724815,19.791111,NaN,33.257963,18.933015,18.643720,44.576687,43.079429,NaN,NaN,31.754851,30.861574
4,2006-05-01,47.074753,20.242859,NaN,33.658806,19.433629,18.963548,46.515144,43.878494,NaN,NaN,32.974386,31.421021


In [4]:
# Inspect columns

print("Master columns:")
print(master_df.columns.tolist())

print("\nCounty price columns:")
print(county_prices.columns.tolist())

print("\nNational price columns:")
print(national_prices.columns.tolist())

Master columns:
['ipc_date', 'analysis_period', 'county', 'max_ipc_phase', 'phase_3_plus_population', 'total_population', 'phase_3_plus_percentage', 'rainfall_date', 'mean_rainfall_mm', 'rainfall_3_month_total', 'rainfall_6_month_total', 'rainfall_3_month_avg', 'rainfall_6_month_avg', 'date', 'mean_ndvi', 'ndvi_1_month_mean', 'ndvi_3_month_mean', 'ndvi_6_month_mean', 'ndvi_anomaly']

County price columns:
['county', 'month', 'county_beans_price_per_kg', 'county_maize_price_per_kg', 'county_rice_price_per_kg', 'county_staple_price_per_kg', 'county_price_available', 'county_staple_items_available', 'county_maize_price_per_kg_3_month_avg', 'county_maize_price_per_kg_6_month_avg', 'county_beans_price_per_kg_3_month_avg', 'county_beans_price_per_kg_6_month_avg', 'county_rice_price_per_kg_3_month_avg', 'county_rice_price_per_kg_6_month_avg', 'county_staple_price_per_kg_3_month_avg', 'county_staple_price_per_kg_6_month_avg']

National price columns:
['month', 'national_beans_price_per_kg', 'n

## **Prepare Date Columns for Merging**

The master dataset uses IPC analysis dates, while the food price feature files use monthly dates.

To merge them correctly, the IPC date will be converted into a month-level date called `month`.

The merge keys will be:

- `county` and `month` for county-level food price features
- `month` only for national food price features

In [5]:
# Convert date columns

master_df["ipc_date"] = pd.to_datetime(master_df["ipc_date"], errors="coerce")
county_prices["month"] = pd.to_datetime(county_prices["month"], errors="coerce")
national_prices["month"] = pd.to_datetime(national_prices["month"], errors="coerce")

# Create month key from IPC date
master_df["month"] = master_df["ipc_date"].dt.to_period("M").dt.to_timestamp()

print("Master month range:", master_df["month"].min(), "to", master_df["month"].max())
print("County price month range:", county_prices["month"].min(), "to", county_prices["month"].max())
print("National price month range:", national_prices["month"].min(), "to", national_prices["month"].max())

master_df[["ipc_date", "month", "analysis_period", "county"]].head()

Master month range: 2019-07-01 00:00:00 to 2026-02-01 00:00:00
County price month range: 2006-01-01 00:00:00 to 2026-04-01 00:00:00
National price month range: 2006-01-01 00:00:00 to 2026-04-01 00:00:00


,ipc_date,month,analysis_period,county
0,2019-07-01,2019-07-01,Jul 2019,Baringo
1,2019-07-01,2019-07-01,Jul 2019,Embu
2,2019-07-01,2019-07-01,Jul 2019,Garissa
3,2019-07-01,2019-07-01,Jul 2019,Isiolo
4,2019-07-01,2019-07-01,Jul 2019,Kajiado


## **Merge County-Level Food Price Features**

County-level food price features are merged into the master dataset using `county` and `month`.

This adds local food price indicators where direct market coverage exists for a target county.

Some counties will not receive county-level price values because they do not have direct mapped market coverage in the WFP food price dataset.

In [6]:
# Merge county-level food price features with master dataset

master_with_county_prices = master_df.merge(
    county_prices,
    on=["county", "month"],
    how="left"
)

print("Before county price merge:", master_df.shape)
print("After county price merge:", master_with_county_prices.shape)

master_with_county_prices.head()

Before county price merge: (322, 20)
After county price merge: (322, 34)


,ipc_date,analysis_period,county,max_ipc_phase,phase_3_plus_population,total_population,phase_3_plus_percentage,rainfall_date,mean_rainfall_mm,rainfall_3_month_total,...,county_price_available,county_staple_items_available,county_maize_price_per_kg_3_month_avg,county_maize_price_per_kg_6_month_avg,county_beans_price_per_kg_3_month_avg,county_beans_price_per_kg_6_month_avg,county_rice_price_per_kg_3_month_avg,county_rice_price_per_kg_6_month_avg,county_staple_price_per_kg_3_month_avg,county_staple_price_per_kg_6_month_avg
0,2019-07-01,Jul 2019,Baringo,4,105555,703697,0.15,2019-06-01,72.692665,143.441454,...,1.0,2.0,52.900000,50.675000,121.733333,117.366667,NaN,NaN,87.316667,94.075000
1,2019-07-01,Jul 2019,Embu,3,32883,219220,0.15,2019-06-01,8.068131,123.946965,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-07-01,Jul 2019,Garissa,4,151183,431950,0.35,2019-06-01,5.417682,55.185197,...,1.0,1.0,66.666667,65.833333,NaN,NaN,NaN,NaN,66.666667,65.833333
3,2019-07-01,Jul 2019,Isiolo,4,54413,155465,0.35,2019-06-01,0.936229,26.591658,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-07-01,Jul 2019,Kajiado,3,43536,870721,0.05,2019-06-01,6.391582,62.716861,...,1.0,2.0,53.333333,51.000000,98.800000,97.066667,NaN,NaN,76.066667,74.033333


## **Merge National Food Price Features**

National food price features are merged using the monthly date only.

These features provide Kenya-wide food price indicators for every county-period record.

This is useful because some target counties do not have direct county-level food price market coverage.

In [7]:
# Merge national-level food price features with master dataset

master_food_prices = master_with_county_prices.merge(
    national_prices,
    on="month",
    how="left"
)

print("Before national price merge:", master_with_county_prices.shape)
print("After national price merge:", master_food_prices.shape)

master_food_prices.head()

Before national price merge: (322, 34)
After national price merge: (322, 46)


,ipc_date,analysis_period,county,max_ipc_phase,phase_3_plus_population,total_population,phase_3_plus_percentage,rainfall_date,mean_rainfall_mm,rainfall_3_month_total,...,national_rice_price_per_kg,national_staple_price_per_kg,national_maize_price_per_kg_3_month_avg,national_maize_price_per_kg_6_month_avg,national_beans_price_per_kg_3_month_avg,national_beans_price_per_kg_6_month_avg,national_rice_price_per_kg_3_month_avg,national_rice_price_per_kg_6_month_avg,national_staple_price_per_kg_3_month_avg,national_staple_price_per_kg_6_month_avg
0,2019-07-01,Jul 2019,Baringo,4,105555,703697,0.15,2019-06-01,72.692665,143.441454,...,NaN,71.969014,47.466957,42.789027,90.120916,82.49896,NaN,NaN,68.793937,62.643993
1,2019-07-01,Jul 2019,Embu,3,32883,219220,0.15,2019-06-01,8.068131,123.946965,...,NaN,71.969014,47.466957,42.789027,90.120916,82.49896,NaN,NaN,68.793937,62.643993
2,2019-07-01,Jul 2019,Garissa,4,151183,431950,0.35,2019-06-01,5.417682,55.185197,...,NaN,71.969014,47.466957,42.789027,90.120916,82.49896,NaN,NaN,68.793937,62.643993
3,2019-07-01,Jul 2019,Isiolo,4,54413,155465,0.35,2019-06-01,0.936229,26.591658,...,NaN,71.969014,47.466957,42.789027,90.120916,82.49896,NaN,NaN,68.793937,62.643993
4,2019-07-01,Jul 2019,Kajiado,3,43536,870721,0.05,2019-06-01,6.391582,62.716861,...,NaN,71.969014,47.466957,42.789027,90.120916,82.49896,NaN,NaN,68.793937,62.643993


## **Food Price Missingness After Merge**

After merging county-level and national-level food price features, missing values are checked.

This helps confirm:

1. Which county-level food price features are available.
2. Whether national food price features successfully cover all county-period records.
3. Which food price columns are safest for modeling.

In [8]:
# Check missing values in food price columns after both merges

food_price_cols = [col for col in master_food_prices.columns if "price" in col]

missing_food_price_summary = (
    master_food_prices[food_price_cols]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

missing_food_price_summary.columns = ["column", "missing_percentage"]

missing_food_price_summary

,column,missing_percentage
0,county_beans_price_per_kg,78.26
1,county_maize_price_per_kg,73.91
2,county_rice_price_per_kg,85.71
3,county_staple_price_per_kg,72.05
4,county_price_available,72.05
5,county_maize_price_per_kg_3_month_avg,72.67
6,county_maize_price_per_kg_6_month_avg,72.36
7,county_beans_price_per_kg_3_month_avg,76.71
8,county_beans_price_per_kg_6_month_avg,76.40
9,county_rice_price_per_kg_3_month_avg,84.78


In [9]:
# Key food price columns missingness

key_price_cols = [
    "county_staple_price_per_kg",
    "county_staple_price_per_kg_3_month_avg",
    "county_staple_price_per_kg_6_month_avg",
    "national_staple_price_per_kg",
    "national_staple_price_per_kg_3_month_avg",
    "national_staple_price_per_kg_6_month_avg",
    "county_price_available"
]

master_food_prices[key_price_cols].isna().sum()

county_staple_price_per_kg                  232
county_staple_price_per_kg_3_month_avg      232
county_staple_price_per_kg_6_month_avg      232
national_staple_price_per_kg                  0
national_staple_price_per_kg_3_month_avg      0
national_staple_price_per_kg_6_month_avg      0
county_price_available                      232
dtype: int64

## **Create Hybrid Food Price Features**

County-level food price features are missing for some county-period records because not every target county has direct market price coverage.

National food price features are available for all county-period records.

To keep the final modeling dataset complete, hybrid food price features are created.

The logic is:

- Use county-level food price values where available.
- If county-level food price values are missing, use national food price values as fallback.

This keeps local market information where possible while still giving every county-period a food price signal.

In [10]:
# Create hybrid food price features
# Use county-level values where available; otherwise use national-level values.

master_food_prices["final_staple_price_per_kg"] = (
    master_food_prices["county_staple_price_per_kg"]
    .fillna(master_food_prices["national_staple_price_per_kg"])
)

master_food_prices["final_staple_price_per_kg_3_month_avg"] = (
    master_food_prices["county_staple_price_per_kg_3_month_avg"]
    .fillna(master_food_prices["national_staple_price_per_kg_3_month_avg"])
)

master_food_prices["final_staple_price_per_kg_6_month_avg"] = (
    master_food_prices["county_staple_price_per_kg_6_month_avg"]
    .fillna(master_food_prices["national_staple_price_per_kg_6_month_avg"])
)

# Create source flag
master_food_prices["food_price_source"] = np.where(
    master_food_prices["county_staple_price_per_kg"].notna(),
    "county",
    "national_proxy"
)

# Create binary flag
master_food_prices["county_food_price_matched"] = np.where(
    master_food_prices["county_staple_price_per_kg"].notna(),
    1,
    0
)

# If county_price_available is missing, set it to 0
master_food_prices["county_price_available"] = (
    master_food_prices["county_price_available"]
    .fillna(0)
    .astype(int)
)

print("Hybrid food price features created.")

master_food_prices[
    [
        "ipc_date",
        "analysis_period",
        "county",
        "county_staple_price_per_kg",
        "national_staple_price_per_kg",
        "final_staple_price_per_kg",
        "food_price_source",
        "county_food_price_matched"
    ]
].head(20)

Hybrid food price features created.


,ipc_date,analysis_period,county,county_staple_price_per_kg,national_staple_price_per_kg,final_staple_price_per_kg,food_price_source,county_food_price_matched
0,2019-07-01,Jul 2019,Baringo,97.45,71.969014,97.450000,county,1
1,2019-07-01,Jul 2019,Embu,NaN,71.969014,71.969014,national_proxy,0
2,2019-07-01,Jul 2019,Garissa,68.00,71.969014,68.000000,county,1
3,2019-07-01,Jul 2019,Isiolo,NaN,71.969014,71.969014,national_proxy,0
4,2019-07-01,Jul 2019,Kajiado,76.70,71.969014,76.700000,county,1
5,2019-07-01,Jul 2019,Kilifi,83.45,71.969014,83.450000,county,1
6,2019-07-01,Jul 2019,Kitui,65.55,71.969014,65.550000,county,1
7,2019-07-01,Jul 2019,Kwale,NaN,71.969014,71.969014,national_proxy,0
8,2019-07-01,Jul 2019,Laikipia,NaN,71.969014,71.969014,national_proxy,0
9,2019-07-01,Jul 2019,Lamu,NaN,71.969014,71.969014,national_proxy,0


In [11]:
# Check missingness in final hybrid food price features

hybrid_price_cols = [
    "final_staple_price_per_kg",
    "final_staple_price_per_kg_3_month_avg",
    "final_staple_price_per_kg_6_month_avg",
    "food_price_source",
    "county_food_price_matched"
]

master_food_prices[hybrid_price_cols].isna().sum()

final_staple_price_per_kg                0
final_staple_price_per_kg_3_month_avg    0
final_staple_price_per_kg_6_month_avg    0
food_price_source                        0
county_food_price_matched                0
dtype: int64

## **Hybrid Food Price Feature Results**

Hybrid food price features were created successfully.

The final food price columns use county-level prices where available and national prices as fallback where county-level prices are missing.

This creates complete food price indicators for every county-period record in the master dataset.

The final hybrid food price columns have no missing values:

- `final_staple_price_per_kg`
- `final_staple_price_per_kg_3_month_avg`
- `final_staple_price_per_kg_6_month_avg`

The column `food_price_source` shows whether each row used a direct county-level price or a national proxy price.

The column `county_food_price_matched` provides a binary flag:

- `1` means county-level food price was matched.
- `0` means national proxy food price was used.

In [12]:
# Check how many rows used county prices vs national proxy

price_source_summary = (
    master_food_prices["food_price_source"]
    .value_counts()
    .reset_index()
)

price_source_summary.columns = ["food_price_source", "records"]

price_source_summary

,food_price_source,records
0,national_proxy,232
1,county,90


In [13]:
# Check percentage distribution of food price source

price_source_percentage = (
    master_food_prices["food_price_source"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .reset_index()
)

price_source_percentage.columns = ["food_price_source", "percentage"]

price_source_percentage

,food_price_source,percentage
0,national_proxy,72.05
1,county,27.95


## **Food Price Source Distribution**

The hybrid food price feature uses county-level prices where available and national prices as fallback where county-level prices are missing.

After applying this logic:

- 90 records used direct county-level food prices.
- 232 records used national proxy food prices.

This means `27.95%` of the master dataset uses county-level food price data, while `72.05%` uses national proxy prices.

This confirms that county-level food price coverage is limited, but the hybrid approach keeps the final food price features complete for all county-period records.

## **Final Dataset Check Before Saving**

The hybrid food price features have now been added to the master modeling dataset.

Before saving the final dataset, this section checks:

1. Final dataset shape
2. Missing values in key food price columns
3. Food price source distribution
4. Main columns added by this notebook

In [14]:
# Final dataset shape check

print("Original master dataset shape:", master_df.shape)
print("Final master dataset with food prices shape:", master_food_prices.shape)

Original master dataset shape: (322, 20)
Final master dataset with food prices shape: (322, 51)


In [15]:
# Check final hybrid food price columns

final_food_price_cols = [
    "final_staple_price_per_kg",
    "final_staple_price_per_kg_3_month_avg",
    "final_staple_price_per_kg_6_month_avg",
    "food_price_source",
    "county_food_price_matched",
    "county_price_available"
]

master_food_prices[final_food_price_cols].isna().sum()

final_staple_price_per_kg                0
final_staple_price_per_kg_3_month_avg    0
final_staple_price_per_kg_6_month_avg    0
food_price_source                        0
county_food_price_matched                0
county_price_available                   0
dtype: int64

In [16]:
# Preview important final food price columns

preview_cols = [
    "ipc_date",
    "analysis_period",
    "county",
    "max_ipc_phase",
    "phase_3_plus_percentage",
    "final_staple_price_per_kg",
    "final_staple_price_per_kg_3_month_avg",
    "final_staple_price_per_kg_6_month_avg",
    "food_price_source",
    "county_food_price_matched"
]

master_food_prices[preview_cols].head(20)

,ipc_date,analysis_period,county,max_ipc_phase,phase_3_plus_percentage,final_staple_price_per_kg,final_staple_price_per_kg_3_month_avg,final_staple_price_per_kg_6_month_avg,food_price_source,county_food_price_matched
0,2019-07-01,Jul 2019,Baringo,4,0.15,97.450000,87.316667,94.075000,county,1
1,2019-07-01,Jul 2019,Embu,3,0.15,71.969014,68.793937,62.643993,national_proxy,0
2,2019-07-01,Jul 2019,Garissa,4,0.35,68.000000,66.666667,65.833333,county,1
3,2019-07-01,Jul 2019,Isiolo,4,0.35,71.969014,68.793937,62.643993,national_proxy,0
4,2019-07-01,Jul 2019,Kajiado,3,0.05,76.700000,76.066667,74.033333,county,1
5,2019-07-01,Jul 2019,Kilifi,3,0.15,83.450000,77.050000,71.241667,county,1
6,2019-07-01,Jul 2019,Kitui,3,0.20,65.550000,63.183333,57.091667,county,1
7,2019-07-01,Jul 2019,Kwale,3,0.15,71.969014,68.793937,62.643993,national_proxy,0
8,2019-07-01,Jul 2019,Laikipia,3,0.10,71.969014,68.793937,62.643993,national_proxy,0
9,2019-07-01,Jul 2019,Lamu,3,0.20,71.969014,68.793937,62.643993,national_proxy,0


In [18]:
# Save final master dataset with food price features

OUTPUT_FILE = PROCESSED_DIR / "ipc_rainfall_ndvi_food_price_master_dataset.csv"

master_food_prices.to_csv(OUTPUT_FILE, index=False)

print("Final master dataset with food price features saved to:")
print(OUTPUT_FILE)

print("Final shape:", master_food_prices.shape)

Final master dataset with food price features saved to:
..\02_data\processed\ipc_rainfall_ndvi_food_price_master_dataset.csv
Final shape: (322, 51)


In [19]:
# Verify saved final dataset

saved_final_master = pd.read_csv(OUTPUT_FILE)

print("Saved final master dataset shape:", saved_final_master.shape)

saved_final_master.head()

Saved final master dataset shape: (322, 51)


,ipc_date,analysis_period,county,max_ipc_phase,phase_3_plus_population,total_population,phase_3_plus_percentage,rainfall_date,mean_rainfall_mm,rainfall_3_month_total,...,national_beans_price_per_kg_6_month_avg,national_rice_price_per_kg_3_month_avg,national_rice_price_per_kg_6_month_avg,national_staple_price_per_kg_3_month_avg,national_staple_price_per_kg_6_month_avg,final_staple_price_per_kg,final_staple_price_per_kg_3_month_avg,final_staple_price_per_kg_6_month_avg,food_price_source,county_food_price_matched
0,2019-07-01,Jul 2019,Baringo,4,105555,703697,0.15,2019-06-01,72.692665,143.441454,...,82.49896,NaN,NaN,68.793937,62.643993,97.450000,87.316667,94.075000,county,1
1,2019-07-01,Jul 2019,Embu,3,32883,219220,0.15,2019-06-01,8.068131,123.946965,...,82.49896,NaN,NaN,68.793937,62.643993,71.969014,68.793937,62.643993,national_proxy,0
2,2019-07-01,Jul 2019,Garissa,4,151183,431950,0.35,2019-06-01,5.417682,55.185197,...,82.49896,NaN,NaN,68.793937,62.643993,68.000000,66.666667,65.833333,county,1
3,2019-07-01,Jul 2019,Isiolo,4,54413,155465,0.35,2019-06-01,0.936229,26.591658,...,82.49896,NaN,NaN,68.793937,62.643993,71.969014,68.793937,62.643993,national_proxy,0
4,2019-07-01,Jul 2019,Kajiado,3,43536,870721,0.05,2019-06-01,6.391582,62.716861,...,82.49896,NaN,NaN,68.793937,62.643993,76.700000,76.066667,74.033333,county,1


## **Final Output Created**

The final master dataset with food price features was saved successfully.

Output file:

`02_data/processed/ipc_rainfall_ndvi_food_price_master_dataset.csv`

This dataset now includes:

- IPC food insecurity outcome variables
- Rainfall features
- NDVI features
- County-level food price features where available
- National food price proxy features
- Final hybrid food price features

The safest food price features for the first modeling version are:

- `final_staple_price_per_kg`
- `final_staple_price_per_kg_3_month_avg`
- `final_staple_price_per_kg_6_month_avg`
- `food_price_source`
- `county_food_price_matched`

This dataset is ready for the next notebook, where model performance can be compared before and after adding food price features.